# Train SASRec (Cross)

In [1]:
import numpy as np

# For NumPy 2.0 compatibility with RecBole 1.2
np.float_ = np.float64
np.int_ = np.int64
np.complex_ = np.complex128
np.unicode_ = np.str_

In [ ]:
from typing import Any
import torch
import pandas as pd
from recbole.config import Config
from recbole.data.dataloader import FullSortEvalDataLoader, AbstractDataLoader
from recbole.data import create_dataset, data_preparation
from recbole.model.sequential_recommender import SASRec
from recbole.trainer import Trainer
from recbole.utils import init_seed, init_logger

In [3]:
# --- Config ---
# Assume we have `*.train.inter`, `*.valid.inter`, `*.test.inter`
DATASET_NAME: str = "cross" 
DATA_DIR: str = "../data"
SEED = 67

# Sequential recommendation config
MAX_ITEM_LIST_LENGTH: int = 50

## Create dataset

In [ ]:
config_dict: dict[str, Any] = {
    "data_path": DATA_DIR,
    "dataset": DATASET_NAME,
    "USER_ID_FIELD": "user_id",
    "ITEM_ID_FIELD": "item_id",
    "RATING_FIELD": "rating",
    "TIME_FIELD": "timestamp",
    "benchmark_filename": ["train", "valid", "test"],
    "load_col": {
        "inter": ["user_id", "item_id", "rating", "timestamp", "item_id_list"],
        "user": ["user_id", "category"],
        "item": ["item_id", "is_target"],
    },
    # This is needed to tell RecBole that `item_id_list` is an alias of `item_id` for sequential recommendation.
    "alias_of_item_id": ["item_id_list"],
    "MAX_ITEM_LIST_LENGTH": MAX_ITEM_LIST_LENGTH,
    "epochs": 5,
    "train_batch_size": 1024,
    "eval_batch_size": 1024,
    # Not relevant for SASRec
    "train_neg_sample_args": None,
    "eval_args": {
        # Split is already determined by the `benchmark filename` as separate `.inter` files
        "split": None, 
        "order": "TO",
        "mode": "full",
    },
    "metrics": ["NDCG", "Recall", "MRR"],
    "valid_metric": "NDCG@10",
    "seed": SEED,
}

config: Config = Config(model="SASRec", config_dict=config_dict)
init_logger(config)

# Use MPS if available (for Apple Silicon)
if torch.backends.mps.is_available():
    print("Configuring MPS (Apple Silicon GPU) for training.")
    config.final_config_dict["device"] = torch.device("mps")
    
init_seed(SEED, reproducibility=True)

In [ ]:
dataset = create_dataset(config)
train_data, valid_data, test_data = data_preparation(config, dataset)

Configuring MPS (Apple Silicon GPU) for training.


/Users/nginyc/repos/amazon-item-recommender/.venv/lib/python3.12/site-packages/recbole/data/dataset/dataset.py:501: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[field].fillna(value="", inplace=True)
/Users/nginyc/repos/amazon-item-recommender/.venv/lib/python3.12/site-packages/recbole/data/dataset/dataset.py:501: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because

## Train SASRec

In [6]:
class MaskedTrainer(Trainer):
    def __init__(self, config, model, item_mask):
        '''
        A Trainer that can mask out certain items (i.e. items in the target category)
            during evaluation.
        '''
        super().__init__(config, model)
        self.item_mask = item_mask

    def _full_sort_batch_eval(self, batched_data):
        interaction, scores, positive_u, positive_i = (
            super()._full_sort_batch_eval(batched_data)
        )
        scores[:, ~self.item_mask.to(scores.device)] = -torch.inf
        return interaction, scores, positive_u, positive_i
    

model: SASRec = SASRec(config, train_data.dataset).to(config["device"])
target_item_mask = dataset.item_feat['is_target'].bool()
trainer: Trainer = MaskedTrainer(config, model, item_mask=target_item_mask)

In [7]:
best_valid_score: float
best_valid_result: dict[str, float]
best_valid_score, best_valid_result = trainer.fit(train_data, valid_data, verbose=True, show_progress=True)

Train     0:   0%|                                                          | 0/782 [00:00<?, ?it/s]/Users/nginyc/repos/amazon-item-recommender/.venv/lib/python3.12/site-packages/recbole/trainer/trainer.py:235: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = amp.GradScaler(enabled=self.enable_scaler)
Train     1:  64%|██████████████████████████████▊                 | 503/782 [06:13<03:13,  1.44it/s]Error: command buffer exited with error status.
	The Metal Performance Shaders operations encoded on it may not have completed.
	Error: 
	(null)
	Caused GPU Hang Error (00000003:kIOGPUCommandBufferCallbackErrorHang)
	<AGXG13XFamilyCommandBuffer: 0xb7c702a00>
    label = <none> 
    device = <AGXG13XDevice: 0x101a94ab0>
        name = Apple M1 Max 
    commandQueue = <AGXG13XFamilyCommandQueue: 0x101a932b0>
        label = <none> 
        device = <AGXG13XDevice: 0x101a94ab0>
            name = Apple M1 

In [8]:
print(f"\nBest valid score: {best_valid_score:.4f}")
print("Best valid result:")
for metric, score in best_valid_result.items():
    print(f"  {metric}: {score:.4f}")


Best valid score: 0.0042
Best valid result:
  ndcg@10: 0.0042
  recall@10: 0.0079
  mrr@10: 0.0031


## Evaluate on test set

In [9]:
def evaluate_on_subset(
    data: AbstractDataLoader,
    mask: np.ndarray,
    label: str
):
    '''
    Evaluate the model on a subset of interactions defined by `mask`.
    '''
    inter_feat = data.dataset.inter_feat
    cat_ds = data.dataset.copy(inter_feat[mask])
    cat_dl = FullSortEvalDataLoader(config, cat_ds, sampler=None, shuffle=False)
    results = trainer.evaluate(cat_dl)
    print(f"\nEvaluation ({label})")
    print(f'-' * 20)
    print(f"  Interactions: {mask.sum()}")
    for metric, val in results.items():
        print(f"  {metric}: {val:.4f}")

In [10]:
test_result: dict[str, float] = trainer.evaluate(test_data)

print("Test results (Overall):")
for metric, value in test_result.items():
    print(f"  {metric}: {value:.4f}")

/Users/nginyc/repos/amazon-item-recommender/.venv/lib/python3.12/site-packages/recbole/trainer/trainer.py:583: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = tor

Test results (Overall):
  ndcg@10: 0.0040
  recall@10: 0.0079
  mrr@10: 0.0029


In [11]:
# Map integer categories to labels
tok = dataset.field2token_id["category"]
CAT_LABELS = {tok["0"]: "warm", tok["1"]: "cold", tok["2"]: "new"}

uid_to_cat = dict(zip(
    dataset.user_feat[dataset.uid_field].numpy(),
    dataset.user_feat["category"].numpy(),
))

uid_array = test_data.dataset.inter_feat[dataset.uid_field].numpy()

for cat_id, cat_label in CAT_LABELS.items():
    cat_uids = {uid for uid, c in uid_to_cat.items() if c == cat_id}
    mask = np.isin(uid_array, list(cat_uids))
    if not mask.any():
        print(f"\n  {cat_label}: no users in test set — skipping")
        continue

    evaluate_on_subset(test_data, mask, cat_label)


Evaluation (warm)
--------------------
  Interactions: 27008
  ndcg@10: 0.0027
  recall@10: 0.0048
  mrr@10: 0.0020

Evaluation (cold)
--------------------
  Interactions: 36791
  ndcg@10: 0.0057
  recall@10: 0.0109
  mrr@10: 0.0041

Evaluation (new)
--------------------
  Interactions: 36200
  ndcg@10: 0.0033
  recall@10: 0.0070
  mrr@10: 0.0022
